# T19 - 债券指数分析

## 项目概述

本项目主要研究债券指数的风险收益特征分析，包括：

### 核心功能
1. **债券指数数据获取**：从数据库获取债券指数基本信息和价格数据
2. **风险收益指标计算**：计算收益率、最大回撤、年化波动率等指标
3. **分组分析**：按收益率和久期进行分组统计
4. **投资组合优化**：跨久期分组进行组合搭配，优化风险收益
5. **三维分析**：生成收益-久期-最大回撤三维分组表

### 应用场景
- 债券投资组合管理
- 风险对冲策略
- 投资决策支持

---

## 1. 环境配置

导入必要的依赖库，配置运行参数。

In [ ]:
# 标准库
import os
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Union
import itertools

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import pool

# 可视化
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 忽略警告
warnings.filterwarnings('ignore')

# 配置中文字体
def setup_chinese_font():
    """配置中文字体支持"""
    font_names = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Zen Hei', 'sans-serif']
    found_font = None
    for font_name in font_names:
        try:
            fm.findfont(font_name, fallback_to_default=False)
            found_font = font_name
            break
        except:
            continue
    
    if found_font:
        plt.rcParams['font.sans-serif'] = [found_font]
        plt.rcParams['axes.unicode_minus'] = False
        print(f"已设置字体: {found_font}")
    else:
        print("警告: 未找到中文字体，图表可能无法正确显示中文")

setup_chinese_font()

# 设置matplotlib非交互式后端
plt.switch_backend('Agg')

print("环境配置完成!")

## 2. 配置参数

定义全局配置参数，包括数据库连接、分析参数等。

In [ ]:
# 从环境变量获取数据库配置（敏感信息）
DB_USER = os.environ.get('DB_USER', 'hz_work')
DB_PASSWORD = os.environ.get('DB_PASSWORD', '')
DB_HOST = os.environ.get('DB_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
DB_PORT = os.environ.get('DB_PORT', '3306')
DB_NAME = os.environ.get('DB_NAME', 'bond')

# 分析参数配置
CONFIG = {
    # 时间周期定义
    'periods': {
        '30天': 30,
        '3个月': 90,
        '半年': 180,
        '1年': 365
    },
    
    # 分组参数
    'return_bin_size': 0.5,      # 收益率分组区间大小 (%)
    'duration_bin_size': 0.5,    # 久期分组区间大小 (年)
    'drawdown_bin_size': 0.5,    # 最大回撤分组区间大小 (%)
    
    # 组合优化参数
    'max_portfolio_size': 10,    # 最大组合规模
    'max_combinations': 10000,   # 最大组合数
    
    # 可视化参数
    'figure_width': 12,
    'figure_height': 8,
    'dpi': 100,
    'plot_style': 'whitegrid',
    'color_palette': 'viridis',
    
    # 输出目录
    'output_dir': './output',
    'plot_dir': './output/portfolio_plots'
}

# 创建输出目录
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['plot_dir'], exist_ok=True)

print("配置参数加载完成!")
print(f"分析周期: {list(CONFIG['periods'].keys())}")

## 3. 数据获取

从数据库获取债券指数基本信息和价格数据。

In [ ]:
def connect_database() -> Tuple[Optional[sqlalchemy.engine.Engine], Optional[sqlalchemy.engine.Connection]]:
    """
    连接到债券数据库
    
    Returns:
        Tuple[Engine, Connection]: 数据库引擎和连接对象
    """
    try:
        connection_string = f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
        engine = sqlalchemy.create_engine(
            connection_string,
            poolclass=sqlalchemy.pool.NullPool
        )
        connection = engine.connect()
        print("数据库连接成功!")
        return engine, connection
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None


def get_bond_indices(connection: sqlalchemy.engine.Connection) -> pd.DataFrame:
    """
    获取债券指数代码和名称
    
    Args:
        connection: 数据库连接
        
    Returns:
        pd.DataFrame: 债券指数信息
    """
    query = """
    SELECT trade_code, index_name, duration 
    FROM bond.indexcode
    WHERE index_name LIKE '%%财富%%' 
    AND duration IS NOT NULL
    """
    df = pd.read_sql(query, connection)
    print(f"获取到 {len(df)} 个债券指数")
    return df


def get_price_data(connection: sqlalchemy.engine.Connection, 
                   trade_code: str, 
                   start_date: str, 
                   end_date: str) -> pd.DataFrame:
    """
    获取指定指数的价格数据
    
    Args:
        connection: 数据库连接
        trade_code: 指数代码
        start_date: 开始日期
        end_date: 结束日期
        
    Returns:
        pd.DataFrame: 价格数据
    """
    query = f"""
    SELECT DT AS date, CLOSE AS price 
    FROM bond.indexcloseprice
    WHERE TRADE_CODE = '{trade_code}'
    AND DT <= '{end_date}' 
    AND DT >= '{start_date}'
    ORDER BY DT
    """
    return pd.read_sql(query, connection)


# 连接数据库
engine, connection = connect_database()

if connection is not None:
    # 获取债券指数信息
    indices_df = get_bond_indices(connection)
    display(indices_df.head(10))

## 4. 数据处理

进行数据清洗和指标计算。

In [ ]:
# ===================== 核心指标计算函数 =====================

def calculate_return(prices: pd.Series) -> Optional[float]:
    """
    计算收益率
    
    Args:
        prices: 价格序列
        
    Returns:
        float: 收益率 (%)
    """
    if len(prices) < 2:
        return None
    return (prices.iloc[-1] / prices.iloc[0] - 1) * 100


def calculate_max_drawdown(prices_series: pd.Series) -> Optional[float]:
    """
    计算最大回撤
    
    Args:
        prices_series: 价格序列
        
    Returns:
        float: 最大回撤 (%)
    """
    if not isinstance(prices_series, pd.Series):
        prices_series = pd.Series(prices_series)
    
    if len(prices_series) < 2:
        return None
    
    prices = prices_series.values
    max_dd = 0.0
    peak = prices[0]
    
    for price in prices:
        if price > peak:
            peak = price
        if peak == 0:
            dd = 0
        else:
            dd = (peak - price) / peak * 100
        max_dd = max(max_dd, dd)
    
    return max_dd


def calculate_volatility(prices_series: pd.Series, annual_factor: int = 252) -> Optional[float]:
    """
    计算年化波动率
    
    Args:
        prices_series: 价格序列
        annual_factor: 年化因子
        
    Returns:
        float: 年化波动率 (%)
    """
    if not isinstance(prices_series, pd.Series):
        prices_series = pd.Series(prices_series)
    
    if len(prices_series) < 2:
        return None
    
    daily_returns = prices_series.pct_change().dropna()
    if daily_returns.empty or len(daily_returns) < 2:
        return None
    
    return daily_returns.std() * np.sqrt(annual_factor) * 100


print("指标计算函数定义完成!")

In [ ]:
# ===================== 主分析函数 =====================

def analyze_bond_indices(connection: sqlalchemy.engine.Connection, 
                         indices_df: pd.DataFrame) -> pd.DataFrame:
    """
    主要分析流程：计算各指数在不同周期的风险收益指标
    
    Args:
        connection: 数据库连接
        indices_df: 指数信息
        
    Returns:
        pd.DataFrame: 分析结果
    """
    today = datetime.now()
    end_date = today.strftime('%Y-%m-%d')
    
    results = []
    
    for _, index in indices_df.iterrows():
        trade_code = index['trade_code']
        index_name = index['index_name']
        duration = index['duration']
        
        for period_name, days in CONFIG['periods'].items():
            start_date = (today - timedelta(days=days)).strftime('%Y-%m-%d')
            
            # 获取价格数据
            price_data = get_price_data(connection, trade_code, start_date, end_date)
            
            if len(price_data) < 2:
                continue
            
            # 计算指标
            returns = calculate_return(price_data['price'])
            max_dd = calculate_max_drawdown(price_data['price'])
            volatility = calculate_volatility(price_data['price'])
            
            results.append({
                'trade_code': trade_code,
                'index_name': index_name,
                'duration': duration,
                'period': period_name,
                'return': returns,
                'max_drawdown': max_dd,
                'volatility': volatility
            })
    
    results_df = pd.DataFrame(results)
    return results_df


# 执行分析
if connection is not None:
    results_df = analyze_bond_indices(connection, indices_df)
    print(f"\n分析完成，共 {len(results_df)} 条记录")
    
    # 保存结果
    output_path = os.path.join(CONFIG['output_dir'], 'bond_indices_results.csv')
    results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"结果已保存至: {output_path}")
    
    display(results_df.head())

## 5. 核心逻辑

包含分组分析、组合优化和三维分析等核心算法。

In [ ]:
# ===================== 分组分析函数 =====================

def create_bins(data: pd.DataFrame, column: str, bin_size: float) -> np.ndarray:
    """
    创建分组区间
    
    Args:
        data: 数据框
        column: 列名
        bin_size: 区间大小
        
    Returns:
        np.ndarray: 分组区间
    """
    if len(data) == 0:
        return np.array([0, bin_size])
    
    min_val = data[column].min() - (data[column].min() % bin_size)
    max_val = data[column].max() + bin_size - (data[column].max() % bin_size)
    return np.arange(min_val, max_val + bin_size/2, bin_size)


def group_analysis(results_df: pd.DataFrame, 
                   group_by: str = 'return', 
                   bin_size: float = 0.5) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    按指定字段进行分组分析
    
    Args:
        results_df: 分析结果
        group_by: 分组字段 ('return' 或 'duration')
        bin_size: 区间大小
        
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: 分组统计结果和处理后数据
    """
    results_df = results_df.copy()
    
    if group_by == 'return':
        bins = create_bins(results_df, 'return', bin_size)
    else:
        bins = create_bins(results_df, 'duration', bin_size)
    
    labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]
    
    if group_by == 'return':
        results_df['group'] = pd.cut(results_df['return'], bins=bins, labels=labels)
    else:
        results_df['group'] = pd.cut(results_df['duration'], bins=bins, labels=labels)
    
    grouped_stats = results_df.groupby(['group', 'period']).agg({
        'return': 'mean',
        'max_drawdown': 'mean',
        'volatility': 'mean',
        'trade_code': 'count'
    }).rename(columns={'trade_code': 'count'})
    
    return grouped_stats, results_df


# 执行分组分析
if not results_df.empty:
    # 收益率分组分析
    return_groups, _ = group_analysis(results_df, group_by='return')
    return_groups.to_csv(os.path.join(CONFIG['output_dir'], 'return_group_analysis.csv'), encoding='utf-8-sig')
    
    # 久期分组分析
    duration_groups, _ = group_analysis(results_df, group_by='duration')
    duration_groups.to_csv(os.path.join(CONFIG['output_dir'], 'duration_group_analysis.csv'), encoding='utf-8-sig')
    
    print("分组分析完成!")
    display(return_groups.head(10))

In [ ]:
# ===================== 投资组合优化 =====================

def optimize_portfolio(results_df: pd.DataFrame, 
                       indices_df: pd.DataFrame,
                       period: str = '1年', 
                       max_combinations: int = 10000) -> Dict:
    """
    跨久期分组进行组合优化
    
    Args:
        results_df: 分析结果
        indices_df: 指数信息
        period: 分析周期
        max_combinations: 最大组合数
        
    Returns:
        Dict: 最优组合结果
    """
    period_data = results_df[results_df['period'] == period].copy()
    
    # 按久期分组
    duration_bins = create_bins(period_data, 'duration', 0.5)
    duration_labels = [f"{duration_bins[i]:.1f}-{duration_bins[i+1]:.1f}" for i in range(len(duration_bins)-1)]
    period_data['duration_group'] = pd.cut(period_data['duration'], bins=duration_bins, labels=duration_labels)
    
    # 计算每个久期组的平均统计
    duration_group_stats = period_data.groupby('duration_group').agg({
        'return': 'mean',
        'max_drawdown': 'mean'
    })
    
    best_combinations = {}
    
    for duration_group in duration_labels:
        if duration_group not in period_data['duration_group'].values:
            continue
        
        group_indices = period_data[period_data['duration_group'] == duration_group]
        
        if len(group_indices) <= 1:
            continue
        
        target_return = duration_group_stats.loc[duration_group, 'return']
        target_drawdown = duration_group_stats.loc[duration_group, 'max_drawdown']
        
        best_combo = None
        best_score = -float('inf')
        
        all_codes = group_indices['trade_code'].unique()
        min_combo_size = min(3, len(all_codes))
        max_combo_size = min(10, len(all_codes))
        
        for combo_size in range(min_combo_size, max_combo_size + 1):
            combos = list(itertools.combinations(range(len(all_codes)), combo_size))
            np.random.shuffle(combos)
            combos = combos[:max_combinations // (max_combo_size - min_combo_size + 1)]
            
            for combo in combos:
                selected_codes = [all_codes[i] for i in combo]
                combo_data = group_indices[group_indices['trade_code'].isin(selected_codes)]
                
                combo_return = combo_data['return'].mean()
                combo_drawdown = combo_data['max_drawdown'].mean()
                
                if combo_return > target_return and combo_drawdown < target_drawdown:
                    score = (combo_return - target_return) / target_return - (target_drawdown - combo_drawdown) / target_drawdown
                    
                    if best_combo is None or score > best_score:
                        best_combo = selected_codes
                        best_score = score
                        best_return = combo_return
                        best_drawdown = combo_drawdown
        
        if best_combo:
            best_names = indices_df[indices_df['trade_code'].isin(best_combo)]['index_name'].tolist()
            best_combinations[duration_group] = {
                'indices': best_combo,
                'names': best_names,
                'return': best_return,
                'max_drawdown': best_drawdown,
                'target_return': target_return,
                'target_drawdown': target_drawdown
            }
    
    return best_combinations


# 执行组合优化
for period in CONFIG['periods'].keys():
    portfolio_results = optimize_portfolio(results_df, indices_df, period=period)
    
    if portfolio_results:
        portfolio_df_list = []
        for group, data in portfolio_results.items():
            portfolio_df_list.append({
                '久期分组': group,
                '指数代码': ','.join(data['indices']),
                '指数名称': ','.join(data['names']),
                '组合收益': data['return'],
                '组合最大回撤': data['max_drawdown'],
                '目标收益': data.get('target_return', ''),
                '目标最大回撤': data.get('target_drawdown', '')
            })
        
        portfolio_df = pd.DataFrame(portfolio_df_list)
        output_path = os.path.join(CONFIG['output_dir'], f'portfolio_optimization_{period}.csv')
        portfolio_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"{period} 投资组合优化结果已保存")

In [ ]:
# ===================== 三维分析 =====================

def three_dimensional_analysis(results_df: pd.DataFrame, period: str = '1年') -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    生成收益-久期-最大回撤三维分组表
    
    Args:
        results_df: 分析结果
        period: 分析周期
        
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: 三维分组表和处理后数据
    """
    period_data = results_df[results_df['period'] == period].copy()
    
    # 收益率分组
    return_bins = create_bins(period_data, 'return', CONFIG['return_bin_size'])
    return_labels = [f"{return_bins[i]:.1f}%-{return_bins[i+1]:.1f}%" for i in range(len(return_bins)-1)]
    period_data['return_group'] = pd.cut(period_data['return'], bins=return_bins, labels=return_labels)
    
    # 久期分组
    duration_bins = create_bins(period_data, 'duration', CONFIG['duration_bin_size'])
    duration_labels = [f"{duration_bins[i]:.1f}-{duration_bins[i+1]:.1f}" for i in range(len(duration_bins)-1)]
    period_data['duration_group'] = pd.cut(period_data['duration'], bins=duration_bins, labels=duration_labels)
    
    # 最大回撤分组
    drawdown_bins = create_bins(period_data, 'max_drawdown', CONFIG['drawdown_bin_size'])
    drawdown_labels = [f"{drawdown_bins[i]:.1f}%-{drawdown_bins[i+1]:.1f}%" for i in range(len(drawdown_bins)-1)]
    period_data['drawdown_group'] = pd.cut(period_data['max_drawdown'], bins=drawdown_bins, labels=drawdown_labels)
    
    # 三维分组统计
    three_dim_stats = period_data.groupby(['return_group', 'duration_group', 'drawdown_group']).size().reset_index(name='count')
    
    pivot_table = three_dim_stats.pivot_table(
        index=['return_group', 'duration_group'],
        columns='drawdown_group',
        values='count',
        fill_value=0
    )
    
    return pivot_table, period_data


# 执行三维分析
for period in CONFIG['periods'].keys():
    three_dim_table, _ = three_dimensional_analysis(results_df, period=period)
    
    if not three_dim_table.empty:
        output_path = os.path.join(CONFIG['output_dir'], f'three_dim_analysis_{period}.csv')
        three_dim_table.to_csv(output_path, encoding='utf-8-sig')
        print(f"{period} 三维分析结果已保存")

print("\n三维分析完成!")
display(three_dim_table.head())

## 6. 执行与测试

主流程执行和结果可视化。

In [ ]:
# ===================== 结果可视化 =====================

def plot_top_indices(results_df: pd.DataFrame, 
                     period: str = '1年', 
                     metric: str = 'return', 
                     top_n: int = 10,
                     save_path: Optional[str] = None):
    """
    绘制Top N指数图表
    
    Args:
        results_df: 分析结果
        period: 分析周期
        metric: 指标
        top_n: 显示数量
        save_path: 保存路径
    """
    period_data = results_df[results_df['period'] == period].copy()
    top_data = period_data.nlargest(top_n, metric)
    
    fig, ax = plt.subplots(figsize=(CONFIG['figure_width'], CONFIG['figure_height']), dpi=CONFIG['dpi'])
    
    colors = sns.color_palette(CONFIG['color_palette'], top_n)
    bars = ax.barh(top_data['index_name'], top_data[metric], color=colors)
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, f"{width:.2f}", 
                ha='left', va='center')
    
    metric_labels = {
        'return': '收益率 (%)',
        'max_drawdown': '最大回撤 (%)',
        'volatility': '年化波动率 (%)',
        'duration': '久期 (年)'
    }
    
    ax.set_title(f"{period} 周期 {metric_labels.get(metric, metric)} Top {top_n} 债券指数", fontsize=14)
    ax.set_xlabel(metric_labels.get(metric, metric), fontsize=12)
    ax.set_ylabel('指数名称', fontsize=12)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=CONFIG['dpi'], bbox_inches='tight')
        print(f"图表已保存: {save_path}")
    
    plt.show()
    plt.close()


def plot_risk_return_scatter(results_df: pd.DataFrame, 
                              period: str = '1年',
                              save_path: Optional[str] = None):
    """
    绘制风险-收益散点图
    
    Args:
        results_df: 分析结果
        period: 分析周期
        save_path: 保存路径
    """
    period_data = results_df[results_df['period'] == period].copy()
    period_data['sharpe'] = period_data['return'] / period_data['volatility']
    
    fig, ax = plt.subplots(figsize=(CONFIG['figure_width'], CONFIG['figure_height']), dpi=CONFIG['dpi'])
    
    scatter = ax.scatter(period_data['volatility'], period_data['return'], 
                        c=period_data['sharpe'], cmap='RdYlGn',
                        alpha=0.7, s=100)
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('收益/波动率比值', fontsize=12)
    
    # 标注Top 5
    top_indices = period_data.nlargest(5, 'sharpe')
    for _, row in top_indices.iterrows():
        ax.annotate(row['index_name'], 
                  (row['volatility'], row['return']),
                  xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax.set_title(f"{period} 周期风险-收益分析", fontsize=14)
    ax.set_xlabel('年化波动率 (%)', fontsize=12)
    ax.set_ylabel('收益率 (%)', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=CONFIG['dpi'], bbox_inches='tight')
        print(f"图表已保存: {save_path}")
    
    plt.show()
    plt.close()


def plot_correlation_matrix(results_df: pd.DataFrame, 
                             period: str = '1年',
                             save_path: Optional[str] = None):
    """
    绘制相关性矩阵热力图
    
    Args:
        results_df: 分析结果
        period: 分析周期
        save_path: 保存路径
    """
    period_data = results_df[results_df['period'] == period]
    corr_matrix = period_data[['return', 'max_drawdown', 'volatility', 'duration']].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8), dpi=CONFIG['dpi'])
    
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, ax=ax)
    
    ax.set_title(f"{period} 周期指标相关性矩阵", fontsize=14)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=CONFIG['dpi'], bbox_inches='tight')
        print(f"图表已保存: {save_path}")
    
    plt.show()
    plt.close()


# 执行可视化
if not results_df.empty:
    # Top指数图表
    plot_top_indices(results_df, period='1年', metric='return', 
                     save_path=os.path.join(CONFIG['plot_dir'], 'top_return_1y.png'))
    
    # 风险收益散点图
    plot_risk_return_scatter(results_df, period='1年',
                             save_path=os.path.join(CONFIG['plot_dir'], 'risk_return_1y.png'))
    
    # 相关性矩阵
    plot_correlation_matrix(results_df, period='1年',
                            save_path=os.path.join(CONFIG['plot_dir'], 'correlation_1y.png'))

In [ ]:
# ===================== 生成交互式HTML汇总表 =====================

def generate_summary_html(results_df: pd.DataFrame, 
                          period: str = '1年',
                          output_path: str = 'summary_table.html'):
    """
    生成交互式HTML汇总表
    
    Args:
        results_df: 分析结果
        period: 分析周期
        output_path: 输出路径
    """
    period_data = results_df[results_df['period'] == period].copy()
    
    # 样式设置
    styled_df = period_data[['trade_code', 'index_name', 'duration', 'return', 'max_drawdown', 'volatility']].copy()
    styled_df.columns = ['指数代码', '指数名称', '久期', '收益率(%)', '最大回撤(%)', '年化波动率(%)']
    
    # 生成HTML
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>债券指数分析汇总表 - {period}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            h1 {{ color: #2c3e50; }}
            table {{ border-collapse: collapse; width: 100%; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #3498db; color: white; }}
            tr:nth-child(even) {{ background-color: #f2f2f2; }}
            tr:hover {{ background-color: #ddd; }}
        </style>
    </head>
    <body>
        <h1>债券指数分析汇总表 - {period}</h1>
        <p>生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        {styled_df.to_html(index=False, classes='summary-table')}
    </body>
    </html>
    """
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML汇总表已生成: {output_path}")


# 生成汇总表
if not results_df.empty:
    generate_summary_html(results_df, period='1年', 
                          output_path=os.path.join(CONFIG['output_dir'], 'summary_table_styled.html'))

In [ ]:
# ===================== 关闭数据库连接 =====================

if connection is not None:
    connection.close()
    print("\n数据库连接已关闭")

print("\n" + "="*50)
print("债券指数分析任务完成!")
print("="*50)

## 7. 工具函数

可复用的辅助函数集合。

In [ ]:
# ===================== 工具函数集合 =====================

def calculate_portfolio_metrics(price_series_list: List[pd.Series], 
                                 weights: List[float]) -> Dict:
    """
    计算组合的绩效指标
    
    Args:
        price_series_list: 价格序列列表
        weights: 权重列表
        
    Returns:
        Dict: 组合绩效指标
    """
    if len(price_series_list) != len(weights):
        raise ValueError("价格序列数量与权重数量不匹配")
    
    # 归一化价格
    normalized_prices = []
    for ps in price_series_list:
        normalized = ps / ps.iloc[0]
        normalized_prices.append(normalized)
    
    # 计算组合净值
    portfolio_value = sum(w * p for w, p in zip(weights, normalized_prices))
    
    return {
        'total_return': calculate_return(portfolio_value),
        'max_drawdown': calculate_max_drawdown(portfolio_value),
        'volatility': calculate_volatility(portfolio_value)
    }


def filter_indices_by_criteria(indices_df: pd.DataFrame, 
                                keywords: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    根据关键词筛选指数
    
    Args:
        indices_df: 指数信息
        keywords: 关键词列表
        
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: 匹配和不匹配的指数
    """
    mask = indices_df['index_name'].apply(
        lambda x: any(kw in str(x) for kw in keywords)
    )
    
    matched = indices_df[mask].copy()
    unmatched = indices_df[~mask].copy()
    
    return matched, unmatched


def export_to_excel(results_dict: Dict[str, pd.DataFrame], 
                    output_path: str):
    """
    将多个DataFrame导出到Excel的不同sheet
    
    Args:
        results_dict: 结果字典 {sheet名: DataFrame}
        output_path: 输出路径
    """
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for sheet_name, df in results_dict.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"Excel文件已导出: {output_path}")


def get_period_statistics(results_df: pd.DataFrame, 
                          period: str) -> Dict:
    """
    获取特定周期的统计摘要
    
    Args:
        results_df: 分析结果
        period: 分析周期
        
    Returns:
        Dict: 统计摘要
    """
    period_data = results_df[results_df['period'] == period]
    
    return {
        'period': period,
        'total_indices': len(period_data),
        'avg_return': period_data['return'].mean(),
        'avg_drawdown': period_data['max_drawdown'].mean(),
        'avg_volatility': period_data['volatility'].mean(),
        'avg_duration': period_data['duration'].mean(),
        'max_return': period_data['return'].max(),
        'min_return': period_data['return'].min(),
    }


print("工具函数定义完成!")

In [ ]:
# ===================== 示例：使用工具函数 =====================

# 获取各周期统计摘要
if not results_df.empty:
    print("\n各周期统计摘要:")
    print("-" * 60)
    
    for period in CONFIG['periods'].keys():
        stats = get_period_statistics(results_df, period)
        print(f"\n{period}:")
        print(f"  指数数量: {stats['total_indices']}")
        print(f"  平均收益率: {stats['avg_return']:.2f}%")
        print(f"  平均最大回撤: {stats['avg_drawdown']:.2f}%")
        print(f"  平均波动率: {stats['avg_volatility']:.2f}%")
        print(f"  平均久期: {stats['avg_duration']:.2f}年")